# Módulos estándar · Testing

Comprobar que el código hace lo que debe con `assert`, `doctest`, `unittest` y `pytest`, escribir los tests antes que el código (TDD) y aislar dependencias externas con `unittest.mock`.

## 1. Comprobaciones con `assert`

Escribir `es_primo` y comprobarla con `assert` contra la lista de primos menores que 30. Ver qué ocurre cuando un `assert` falla. Ojo: con `python -O` los `assert` se desactivan, así que no sirven para validar datos de entrada.

In [1]:
def es_primo(n):
    if n < 2:
        return False
    i = 2
    while i * i <= n:
        if n % i == 0:
            return False
        i += 1
    return True

assert [n for n in range(30) if es_primo(n)] == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert not es_primo(1) and not es_primo(-7)

try:
    assert es_primo(9), "9 debería ser primo"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: 9 debería ser primo


## 2. `doctest`

Documentar `factorial` con ejemplos en el docstring (incluida una excepción) y comprobarlos con `doctest.run_docstring_examples` y `doctest.testmod`. Ver cómo se informa de un ejemplo incorrecto.

In [2]:
import doctest

def factorial(n):
    """Devuelve n!.

    >>> factorial(0)
    1
    >>> [factorial(i) for i in range(6)]
    [1, 1, 2, 6, 24, 120]
    >>> factorial(-1)
    Traceback (most recent call last):
        ...
    ValueError: n debe ser >= 0
    """
    if n < 0:
        raise ValueError("n debe ser >= 0")
    resultado = 1
    for i in range(2, n + 1):
        resultado *= i
    return resultado

doctest.run_docstring_examples(factorial, globals(), verbose=True)
print(doctest.testmod())

Finding tests in NoName
Trying:
    factorial(0)
Expecting:
    1
ok
Trying:
    [factorial(i) for i in range(6)]
Expecting:
    [1, 1, 2, 6, 24, 120]
ok
Trying:
    factorial(-1)
Expecting:
    Traceback (most recent call last):
        ...
    ValueError: n debe ser >= 0
ok
TestResults(failed=0, attempted=3)


In [3]:
def media(numeros):
    """
    >>> media([1, 2, 3, 4])
    2.5
    >>> media([1, 2])
    1
    """
    return sum(numeros) / len(numeros)

doctest.run_docstring_examples(media, globals(), name="media")

**********************************************************************
File "__main__", line 5, in media
Failed example:
    media([1, 2])
Expected:
    1
Got:
    1.5


## 3. Primer `TestCase` con `unittest`

Escribir los tests de `es_primo` en una clase que herede de `unittest.TestCase` (con `assertTrue`, `assertFalse`, `assertEqual` y `subTest`) y ejecutarlos con `unittest.main(argv=[""], exit=False)`.

In [4]:
import unittest

class TestEsPrimo(unittest.TestCase):
    def test_primos(self):
        for n in (2, 3, 5, 7, 11, 97):
            with self.subTest(n=n):
                self.assertTrue(es_primo(n))

    def test_no_primos(self):
        for n in (-3, 0, 1, 4, 9, 91):
            with self.subTest(n=n):
                self.assertFalse(es_primo(n))

    def test_primos_hasta_30(self):
        self.assertEqual([n for n in range(30) if es_primo(n)], [2, 3, 5, 7, 11, 13, 17, 19, 23, 29])

programa = unittest.main(argv=[""], exit=False, verbosity=2)
print(programa.result.wasSuccessful())

test_no_primos (__main__.TestEsPrimo.test_no_primos) ... 

ok


test_primos (__main__.TestEsPrimo.test_primos) ... 

ok


test_primos_hasta_30 (__main__.TestEsPrimo.test_primos_hasta_30) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


True


## 4. `setUp` y `assertRaises`

Clase `CuentaBancaria` con `ingresar` y `retirar`, que lanzan `ValueError` si la cantidad no es positiva y `SaldoInsuficiente` si no hay saldo. Crear una cuenta nueva para cada test en `setUp`, comprobar los errores con `assertRaises` y ejecutar solo esta clase con un `TextTestRunner`.

In [5]:
import sys

class SaldoInsuficiente(Exception):
    pass

class CuentaBancaria:
    def __init__(self, saldo=0):
        self.saldo = saldo

    def ingresar(self, cantidad):
        if cantidad <= 0:
            raise ValueError("la cantidad debe ser positiva")
        self.saldo += cantidad

    def retirar(self, cantidad):
        if cantidad <= 0:
            raise ValueError("la cantidad debe ser positiva")
        if cantidad > self.saldo:
            raise SaldoInsuficiente(f"saldo {self.saldo}, se piden {cantidad}")
        self.saldo -= cantidad

class TestCuentaBancaria(unittest.TestCase):
    def setUp(self):
        self.cuenta = CuentaBancaria(100)

    def test_ingresar(self):
        self.cuenta.ingresar(50)
        self.assertEqual(self.cuenta.saldo, 150)

    def test_retirar(self):
        self.cuenta.retirar(30)
        self.assertEqual(self.cuenta.saldo, 70)

    def test_cantidad_no_positiva(self):
        with self.assertRaises(ValueError):
            self.cuenta.ingresar(0)
        with self.assertRaises(ValueError):
            self.cuenta.retirar(-5)

    def test_saldo_insuficiente(self):
        with self.assertRaises(SaldoInsuficiente) as contexto:
            self.cuenta.retirar(500)
        self.assertIn("500", str(contexto.exception))
        self.assertEqual(self.cuenta.saldo, 100)

suite = unittest.TestLoader().loadTestsFromTestCase(TestCuentaBancaria)
resultado = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
print(resultado.wasSuccessful())

test_cantidad_no_positiva (__main__.TestCuentaBancaria.test_cantidad_no_positiva) ... 

ok


test_ingresar (__main__.TestCuentaBancaria.test_ingresar) ... 

ok


test_retirar (__main__.TestCuentaBancaria.test_retirar) ... 

ok


test_saldo_insuficiente (__main__.TestCuentaBancaria.test_saldo_insuficiente) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.002s

OK


True


## 5. `pytest`: módulo y tests

Escribir el módulo `texto.py` y su fichero de tests `test_texto.py` junto al notebook con `%%writefile` y ejecutar `pytest` dentro del propio proceso con `pytest.main`. Comprobar las excepciones con `pytest.raises`.

In [6]:
import os
import pytest

sys.dont_write_bytecode = True  # sin __pycache__ junto al notebook

def ejecuta_pytest(*argumentos):
    # olvida los módulos de esta carpeta ya importados para leer los ficheros recién escritos
    carpeta = os.getcwd()
    for nombre, modulo in list(sys.modules.items()):
        if os.path.dirname(getattr(modulo, "__file__", None) or "") == carpeta:
            del sys.modules[nombre]
    return pytest.main(["-p", "no:cacheprovider", "--color=no", *argumentos])

In [7]:
%%writefile texto.py
def es_palindromo(frase):
    letras = [c.lower() for c in frase if c.isalnum()]
    return letras == letras[::-1]


def cuenta_palabras(frase):
    if not isinstance(frase, str):
        raise TypeError("se esperaba un str")
    return len(frase.split())

Overwriting texto.py


In [8]:
%%writefile test_texto.py
import pytest

from texto import cuenta_palabras, es_palindromo


def test_palindromo():
    assert es_palindromo("Anita lava la tina")
    assert not es_palindromo("Python")


def test_cuenta_palabras():
    assert cuenta_palabras("hola  mundo cruel") == 3
    assert cuenta_palabras("") == 0


def test_cuenta_palabras_tipo_incorrecto():
    with pytest.raises(TypeError, match="str"):
        cuenta_palabras(42)

Overwriting test_texto.py


In [9]:
ejecuta_pytest("-v", "test_texto.py")

============================= test session starts =============================
platform win32 -- Python 3.13.15, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\hecto\Dev\logic-programming-exercises\.venv\Scripts\python.exe
rootdir: C:\Users\hecto\Dev\logic-programming-exercises\src\10-modulos-estandar
plugins: anyio-4.16.0, platformdirs-4.14.5, typeguard-4.6.0
collecting ... 

collected 3 items

test_texto.py::test_palindromo 

PASSED                                    [ 33%]


test_texto.py::test_cuenta_palabras 

PASSED                               [ 66%]


test_texto.py::test_cuenta_palabras_tipo_incorrecto 

PASSED               [100%]



============================== 3 passed in 0.01s ==============================


<ExitCode.OK: 0>

## 6. Fixtures y `parametrize`

Probar una clase `Carrito` usando una *fixture* que prepara un carrito con productos para cada test, `@pytest.mark.parametrize` para repetir un test con varios datos y `pytest.raises` para los descuentos no válidos.

In [10]:
%%writefile carrito.py
class Carrito:
    def __init__(self):
        self.lineas = {}

    def agrega(self, producto, precio, cantidad=1):
        if precio < 0 or cantidad <= 0:
            raise ValueError("precio o cantidad no válidos")
        _, previa = self.lineas.get(producto, (precio, 0))
        self.lineas[producto] = (precio, previa + cantidad)

    def total(self, descuento=0):
        if not 0 <= descuento <= 100:
            raise ValueError("descuento fuera de rango")
        bruto = sum(precio * cantidad for precio, cantidad in self.lineas.values())
        return round(bruto * (100 - descuento) / 100, 2)

Overwriting carrito.py


In [11]:
%%writefile test_carrito.py
import pytest

from carrito import Carrito


@pytest.fixture
def carrito():
    c = Carrito()
    c.agrega("pan", 1.20, 2)
    c.agrega("leche", 0.95, 4)
    return c


def test_total(carrito):
    assert carrito.total() == 6.20


def test_agrega_mismo_producto(carrito):
    carrito.agrega("pan", 1.20)
    assert carrito.lineas["pan"] == (1.20, 3)


def test_carrito_vacio():
    assert Carrito().total() == 0


@pytest.mark.parametrize("descuento, esperado", [(0, 6.20), (10, 5.58), (50, 3.10), (100, 0)])
def test_descuentos(carrito, descuento, esperado):
    assert carrito.total(descuento) == esperado


@pytest.mark.parametrize("descuento", [-1, 101])
def test_descuento_no_valido(carrito, descuento):
    with pytest.raises(ValueError, match="rango"):
        carrito.total(descuento)

Overwriting test_carrito.py


In [12]:
ejecuta_pytest("-v", "test_carrito.py")

============================= test session starts =============================
platform win32 -- Python 3.13.15, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\hecto\Dev\logic-programming-exercises\.venv\Scripts\python.exe
rootdir: C:\Users\hecto\Dev\logic-programming-exercises\src\10-modulos-estandar
plugins: anyio-4.16.0, platformdirs-4.14.5, typeguard-4.6.0
collecting ... 

collected 9 items

test_carrito.py::test_total 

PASSED                                       [ 11%]


test_carrito.py::test_agrega_mismo_producto 

PASSED                       [ 22%]


test_carrito.py::test_carrito_vacio 

... (11 outputs left out)


test_carrito.py::test_descuento_no_valido[101] 

PASSED                    [100%]



============================== 9 passed in 0.02s ==============================


<ExitCode.OK: 0>

## 7. TDD: números romanos

Desarrollar `a_romano(n)` (de 1 a 3999) con TDD: escribir primero los tests y verlos fallar (rojo), escribir el código mínimo que los pasa (verde) y mejorarlo sin romper los tests (refactorizar).

In [13]:
%%writefile test_romanos.py
import pytest

from romanos import a_romano


@pytest.mark.parametrize("n, esperado", [
    (1, "I"), (3, "III"), (4, "IV"), (9, "IX"), (14, "XIV"),
    (40, "XL"), (90, "XC"), (400, "CD"), (1994, "MCMXCIV"), (3999, "MMMCMXCIX"),
])
def test_a_romano(n, esperado):
    assert a_romano(n) == esperado


@pytest.mark.parametrize("n", [0, -5, 4000])
def test_fuera_de_rango(n):
    with pytest.raises(ValueError):
        a_romano(n)

Overwriting test_romanos.py


Rojo: una primera versión ingenua que solo sabe repetir `I`.

In [14]:
%%writefile romanos.py
def a_romano(n):
    return "I" * n

Overwriting romanos.py


In [15]:
ejecuta_pytest("-q", "--tb=no", "test_romanos.py")

.

.

F

F

F

F

... (6 outputs left out)

F

                                                            [100%]


=========================== short test summary info ===========================
FAILED test_romanos.py::test_a_romano[4-IV] - AssertionError: assert 'IIII' =...
FAILED test_romanos.py::test_a_romano[9-IX] - AssertionError: assert 'IIIIIII...
FAILED test_romanos.py::test_a_romano[14-XIV] - AssertionError: assert 'IIIII...
FAILED test_romanos.py::test_a_romano[40-XL] - AssertionError: assert 'IIIIII...
FAILED test_romanos.py::test_a_romano[90-XC] - AssertionError: assert 'IIIIII...
FAILED test_romanos.py::test_a_romano[400-CD] - AssertionError: assert 'IIIII...
FAILED test_romanos.py::test_a_romano[1994-MCMXCIV] - AssertionError: assert ...
FAILED test_romanos.py::test_a_romano[3999-MMMCMXCIX] - AssertionError: asser...
FAILED test_romanos.py::test_fuera_de_rango[0] - Failed: DID NOT RAISE ValueE...
FAILED test_romanos.py::test_fuera_de_rango[-5] - Failed: DID NOT RAISE Value...
FAILED test_romanos.py::test_fuera_de_rango[4000] - Failed: DID NOT RAISE Val...
11 failed, 2 passed in 0.03s

<ExitCode.TESTS_FAILED: 1>

Verde: tabla de valores (incluidos los restos como `CM` o `IV`) de mayor a menor, restando mientras quepa.

In [16]:
%%writefile romanos.py
VALORES = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"),
           (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]


def a_romano(n):
    if not 1 <= n <= 3999:
        raise ValueError("solo de 1 a 3999")
    resultado = ""
    for valor, letras in VALORES:
        while n >= valor:
            resultado += letras
            n -= valor
    return resultado

Overwriting romanos.py


In [17]:
ejecuta_pytest("-q", "test_romanos.py")

.

.

.

.

.

.

... (6 outputs left out)

.

                                                            [100%]


13 passed in 0.02s


<ExitCode.OK: 0>

Refactorizar: `divmod` en lugar del `while` y `join` en lugar de concatenar. Los tests siguen en verde y además los 3999 números dan cadenas distintas.

In [18]:
%%writefile romanos.py
VALORES = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"),
           (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]


def a_romano(n):
    if not 1 <= n <= 3999:
        raise ValueError("solo de 1 a 3999")
    partes = []
    for valor, letras in VALORES:
        veces, n = divmod(n, valor)
        partes.append(letras * veces)
    return "".join(partes)

Overwriting romanos.py


In [19]:
ejecuta_pytest("-q", "test_romanos.py")

from romanos import a_romano

assert len({a_romano(n) for n in range(1, 4000)}) == 3999
print(a_romano(2024), a_romano(1492))

.

.

.

.

.

.

... (6 outputs left out)

.

                                                            [100%]


13 passed in 0.02s


MMXXIV MCDXCII


## 8. Mocks con `unittest.mock`

`informe_tiempo` consulta un servicio externo que no se puede usar en los tests. Sustituirlo por un `Mock`: fijar la respuesta con `return_value`, simular un fallo con `side_effect` y comprobar cómo se llamó. Después, parchear `random.randint` con `patch` para que una tirada de dados sea predecible, también como decorador en un `TestCase`.

In [20]:
from unittest.mock import Mock, patch

def informe_tiempo(ciudad, cliente):
    try:
        datos = cliente.consulta(ciudad)
    except ConnectionError:
        return f"{ciudad}: sin datos"
    return f"{ciudad}: {datos['temperatura']} °C, {datos['cielo']}"

cliente = Mock()
cliente.consulta.return_value = {"temperatura": 24, "cielo": "despejado"}
print(informe_tiempo("Las Palmas", cliente))
cliente.consulta.assert_called_once_with("Las Palmas")

cliente.consulta.side_effect = ConnectionError("sin red")
print(informe_tiempo("La Laguna", cliente))
print(cliente.consulta.call_count, cliente.consulta.call_args_list)

Las Palmas: 24 °C, despejado
La Laguna: sin datos
2 [call('Las Palmas'), call('La Laguna')]


In [21]:
import random

def tirada(dados):
    return sum(random.randint(1, 6) for _ in range(dados))

with patch("random.randint", return_value=6) as falso:
    assert tirada(3) == 18
    print("llamadas:", falso.call_count)

with patch("random.randint", side_effect=[1, 2, 3]):
    assert tirada(3) == 6

print("tirada real:", tirada(3))

class TestTirada(unittest.TestCase):
    @patch("random.randint", return_value=1)
    def test_minimo(self, falso):
        self.assertEqual(tirada(4), 4)
        self.assertEqual(falso.call_count, 4)

suite = unittest.TestLoader().loadTestsFromTestCase(TestTirada)
resultado = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
print(resultado.wasSuccessful())

llamadas: 3
tirada real: 9
test_minimo (__main__.TestTirada.test_minimo) ... 

ok


----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


True
